In [6]:
import pandas as pd
import numpy as np
import joblib
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils.rnn import pad_sequence

In [7]:
session_sequences = pd.read_pickle("session_sequences.pkl")
item_encoder = joblib.load("item_encoder.pkl")

In [10]:
import torch
import torch.nn as nn

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

In [11]:
import torch_geometric

print("PyTorch Geometric Version:", torch_geometric.__version__)

PyTorch Geometric Version: 2.8.0.post1


In [13]:
graph_list = []

for _, row in session_sequences.iterrows():

    sequence = row["ItemSequence"]
    label = row["Label"]

    if len(sequence) < 2:
        continue

    # Unique nodes in the session
    unique_nodes = list(dict.fromkeys(sequence))

    node_map = {node: idx for idx, node in enumerate(unique_nodes)}

    # Node features
    x = torch.tensor(unique_nodes, dtype=torch.long).view(-1, 1)

    # Build edges using node indices
    edges = []

    for i in range(len(sequence) - 1):
        src = node_map[sequence[i]]
        dst = node_map[sequence[i + 1]]
        edges.append([src, dst])

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    y = torch.tensor([label], dtype=torch.float)

    graph = Data(
        x=x,
        edge_index=edge_index,
        y=y
    )

    graph_list.append(graph)

print("Graphs:", len(graph_list))
print(graph_list[0])

Graphs: 250033
Data(x=[4, 1], edge_index=[2, 3], y=[1])


In [14]:
from sklearn.model_selection import train_test_split

labels = [graph.y.item() for graph in graph_list]

train_graphs, temp_graphs = train_test_split(
    graph_list,
    test_size=0.30,
    random_state=42,
    stratify=labels
)

temp_labels = [graph.y.item() for graph in temp_graphs]

val_graphs, test_graphs = train_test_split(
    temp_graphs,
    test_size=0.50,
    random_state=42,
    stratify=temp_labels
)

print("Train:", len(train_graphs))
print("Validation:", len(val_graphs))
print("Test:", len(test_graphs))

Train: 175023
Validation: 37505
Test: 37505


In [25]:
from torch_geometric.loader import DataLoader

BATCH_SIZE = 128

train_loader = DataLoader(
    train_graphs,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_graphs,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_graphs,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [26]:
batch = next(iter(train_loader))

print(batch)

DataBatch(x=[348, 1], edge_index=[2, 342], y=[128], batch=[348], ptr=[129])


In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import GCNConv, global_mean_pool

In [36]:
class GCNPurchasePredictor(nn.Module):

    def __init__(self, vocab_size, embedding_dim=64, hidden_dim=64):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.conv1 = GCNConv(embedding_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

        self.dropout = nn.Dropout(0.3)

        self.fc1 = nn.Linear(hidden_dim, 32)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, data):

        x = data.x.view(-1)

        x = self.embedding(x)

        x = self.conv1(x, data.edge_index)
        x = F.relu(x)

        x = self.conv2(x, data.edge_index)
        x = F.relu(x)

        x = global_mean_pool(x, data.batch)

        x = self.dropout(x)

        x = F.relu(self.fc1(x))

        x = self.fc2(x)

        return x.squeeze(-1)

In [37]:
VOCAB_SIZE = len(item_encoder.classes_) + 1

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = GCNPurchasePredictor(
    vocab_size=VOCAB_SIZE,
    embedding_dim=64,
    hidden_dim=64
).to(device)

print(model)

GCNPurchasePredictor(
  (embedding): Embedding(21129, 64)
  (conv1): GCNConv(64, 64)
  (conv2): GCNConv(64, 64)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc1): Linear(in_features=64, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=1, bias=True)
)


In [38]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

train_labels = np.array([g.y.item() for g in train_graphs])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_labels.astype(int)
)

pos_weight = torch.tensor(
    class_weights[1],
    dtype=torch.float
).to(device)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [39]:
print(model)

GCNPurchasePredictor(
  (embedding): Embedding(21129, 64)
  (conv1): GCNConv(64, 64)
  (conv2): GCNConv(64, 64)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc1): Linear(in_features=64, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=1, bias=True)
)


In [40]:
import copy

EPOCHS = 20
PATIENCE = 3
MIN_DELTA = 0.001

best_val_loss = float("inf")
best_model = None
patience_counter = 0

train_losses = []
val_losses = []

for epoch in range(EPOCHS):

    # -------------------------
    # Training
    # -------------------------
    model.train()

    running_loss = 0

    for batch in train_loader:

        batch = batch.to(device)

        optimizer.zero_grad()

        outputs = model(batch)

        loss = criterion(outputs, batch.y)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)

    # -------------------------
    # Validation
    # -------------------------
    model.eval()

    running_val_loss = 0

    with torch.no_grad():

        for batch in val_loader:

            batch = batch.to(device)

            outputs = model(batch)

            loss = criterion(outputs, batch.y)

            running_val_loss += loss.item()

    val_loss = running_val_loss / len(val_loader)
    val_losses.append(val_loss)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

    # -------------------------
    # Early Stopping
    # -------------------------
    if val_loss < best_val_loss - MIN_DELTA:

        best_val_loss = val_loss
        best_model = copy.deepcopy(model.state_dict())
        patience_counter = 0

        print("✓ Best model saved")

    else:

        patience_counter += 1

        print(f"No improvement ({patience_counter}/{PATIENCE})")

        if patience_counter >= PATIENCE:

            print("Early stopping triggered!")
            break

Epoch 1/20 | Train Loss: 0.8739 | Val Loss: 0.8621
✓ Best model saved
Epoch 2/20 | Train Loss: 0.8189 | Val Loss: 0.8597
✓ Best model saved
Epoch 3/20 | Train Loss: 0.7841 | Val Loss: 0.8560
✓ Best model saved
Epoch 4/20 | Train Loss: 0.7569 | Val Loss: 0.8501
✓ Best model saved
Epoch 5/20 | Train Loss: 0.7355 | Val Loss: 0.8764
No improvement (1/3)
Epoch 6/20 | Train Loss: 0.7179 | Val Loss: 0.9078
No improvement (2/3)
Epoch 7/20 | Train Loss: 0.7017 | Val Loss: 0.9264
No improvement (3/3)
Early stopping triggered!


In [41]:
model.load_state_dict(best_model)

torch.save(
    model.state_dict(),
    "best_gcn_purchase_intent.pth"
)

print("Best GCN model saved.")

Best GCN model saved.


In [42]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

model.eval()

all_labels = []
all_predictions = []
all_probabilities = []

with torch.no_grad():

    for batch in test_loader:

        batch = batch.to(device)

        outputs = model(batch)

        probabilities = torch.sigmoid(outputs)

        predictions = (probabilities >= 0.5).float()

        all_labels.extend(batch.y.cpu().numpy())
        all_predictions.extend(predictions.cpu().numpy())
        all_probabilities.extend(probabilities.cpu().numpy())

accuracy = accuracy_score(all_labels, all_predictions)
precision = precision_score(all_labels, all_predictions)
recall = recall_score(all_labels, all_predictions)
f1 = f1_score(all_labels, all_predictions)
roc_auc = roc_auc_score(all_labels, all_probabilities)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("ROC AUC  :", roc_auc)

print("\nConfusion Matrix")
print(confusion_matrix(all_labels, all_predictions))

print("\nClassification Report")
print(classification_report(all_labels, all_predictions))

Accuracy : 0.7874416744434075
Precision: 0.156614163145089
Recall   : 0.46876197776926026
F1 Score : 0.23478594739873296
ROC AUC  : 0.7198148187445532

Confusion Matrix
[[28310  6586]
 [ 1386  1223]]

Classification Report
              precision    recall  f1-score   support

         0.0       0.95      0.81      0.88     34896
         1.0       0.16      0.47      0.23      2609

    accuracy                           0.79     37505
   macro avg       0.55      0.64      0.56     37505
weighted avg       0.90      0.79      0.83     37505

